In [ ]:
!nvidia-smi

Thu Oct  1 18:31:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers[torch] datasets

In [ ]:
!pip install bitsandbytes trl peft

In [ ]:
from datasets import load_dataset
dataset = load_dataset("HuggingFaceH4/ultrachat_200k")
print(dataset["test_sft"][0])

{'prompt': 'How does the author propose to fix the problem of science alienation in our educational system? What changes does she suggest should be made to science education? Answer according to: Science education should be split into two tracks.\nSplit K-12 science education into two tracks, for majors and nonmajors.\nThose who want to specialize in science could take math and complex chemistry. Nonmajors would focus on science of the everyday—things like kitchen chemistry and CSI-style crime investigations.\nSome years ago, when I was working as a newspaper science writer in California, I fell into a rather idle conversation with a physicist on the subject of science education. Idle for him, at least, because what he said—the way he defined the American system of teaching K-12 students about science—has stayed with me since.\nThis conversation has returned to me many times over the years. On the day my older son, who spent his childhood joyfully chasing insects and reading natural hi

In [ ]:
from datasets import DatasetDict
indices = range(0,100)
dataset_dict = {"train": dataset["train_sft"].select(indices),
                "test": dataset["test_sft"].select(indices)}

raw_dataset = DatasetDict(dataset_dict)
raw_dataset



DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 100
    })
    test: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 100
    })
})

In [ ]:
example = raw_dataset["train"][1]
print(example)

{'prompt': 'Which famous landmarks should I visit in London, beyond the usual ones?', 'prompt_id': 'f5025bdcae61bb77fd98a4d6cd6ba8e0199a098cfebcf6830f4a85e0d13a9e21', 'messages': [{'content': 'Which famous landmarks should I visit in London, beyond the usual ones?', 'role': 'user'}, {'content': "1. Leadenhall Market - a beautiful indoor market with stunning Victorian architecture, also used as a filming location in the Harry Potter films.\n\n2. St. Dunstan in the East - a ruined church in the middle of the city that has been turned into a beautiful public garden.\n\n3. The Monument - a 202-foot-tall column commemorating the Great Fire of London, with a staircase leading to a viewing platform offering great views of the city.\n\n4. The Camden Town Markets - an eclectic collection of markets offering food, fashion, and vintage items, plus live music and street performers.\n\n5. Novelist's House - the former home of Charles Dickens, now a museum dedicated to his life and works.\n\n6. The 

In [ ]:
print(example)

{'prompt': 'Which famous landmarks should I visit in London, beyond the usual ones?', 'prompt_id': 'f5025bdcae61bb77fd98a4d6cd6ba8e0199a098cfebcf6830f4a85e0d13a9e21', 'messages': [{'content': 'Which famous landmarks should I visit in London, beyond the usual ones?', 'role': 'user'}, {'content': "1. Leadenhall Market - a beautiful indoor market with stunning Victorian architecture, also used as a filming location in the Harry Potter films.\n\n2. St. Dunstan in the East - a ruined church in the middle of the city that has been turned into a beautiful public garden.\n\n3. The Monument - a 202-foot-tall column commemorating the Great Fire of London, with a staircase leading to a viewing platform offering great views of the city.\n\n4. The Camden Town Markets - an eclectic collection of markets offering food, fashion, and vintage items, plus live music and street performers.\n\n5. Novelist's House - the former home of Charles Dickens, now a museum dedicated to his life and works.\n\n6. The 

In [ ]:
from transformers import AutoTokenizer
model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

defautl_chat_template = """{% for message in messages %}
                            \n {% if message['role'] == 'user' %}\n
                            {{ '<|user|>\n' + message['content'] + eos_token }}\n
                            {% elif message['role'] == 'system' %}\n
                            {{ '<|system|>\n' + message['content'] + eos_token }}\n
                            {% elif message['role'] == 'assistant' %}\n
                            {{ '<|assistant|>\n'  + message['content'] + eos_token }}\n
                            {% endif %}\n{% if loop.last and add_generation_prompt %}\n
                            {{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"""

tokenizer.chat_template = defautl_chat_template


In [ ]:
import re
import random
from multiprocessing import cpu_count

def apply_chat_template(example, tokenizer):
    messages = example["messages"]
    # We add an empty system message if there is none
    if messages[0]["role"] != "system":
        messages.insert(0, {"role": "system", "content": ""})
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)

    return example

apply_chat_template(example, tokenizer)

In [ ]:
column_names = list(raw_dataset["train"].features)
raw_datasets = raw_dataset.map(apply_chat_template,
                                num_proc=cpu_count(),
                                fn_kwargs={"tokenizer": tokenizer},
                                remove_columns=column_names,
                                desc="Applying chat template",)


Applying chat template (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

Applying chat template (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:

train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

for index in random.sample(range(len(raw_datasets["train"])), 3):
  print(f"Sample {index} of the processed training set:\n\n{raw_datasets['train'][index]['text']}")

In [ ]:
from transformers import BitsAndBytesConfig
import torch

# specify how to quantize the model
# T4 has no bfloat16 support -> use float16
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
)
device_map = {"": torch.cuda.current_device()} if torch.cuda.is_available() else None


In [ ]:
from transformers import AutoModelForCausalLM

model_kwargs = dict(
    dtype=torch.float16,
    use_cache=False, # set to False as we're going to use gradient checkpointing
    device_map=device_map,
    quantization_config=quantization_config,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    **model_kwargs,
)

In [ ]:
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

training_args = SFTConfig(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=1,
    fp16=True,                  # T4: fp16, not bf16
    bf16=False,
    dataset_text_field="text",
    max_length=1024,            # 2048 risks OOM on a 15 GB T4
    packing=False,
    report_to="none",
)

peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainer = SFTTrainer(
        model=model,                 # the 4-bit model loaded above, not the name
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,  # was `tokenizer=`
        peft_config=peft_config,
    )

# fp16 mixed precision can't unscale bf16 grads -> keep trainable (LoRA) params in fp32
for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

In [ ]:
trainer.train()